# 01 · Cross-Team Deep Agent Chat UI

Business Operations와 Customer Operations의 lead agent가 각자 specialist에게 위임하고, typed handoff로 서로 인터페이스하는 과정을 채팅으로 확인합니다. 기본 구현은 안전한 in-memory runtime이며, 각 메시지는 실제 receiver handler를 거쳐 다음 agent로 전달됩니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124;">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️ UI 실행</div>
  <h2 style="margin: 8px 0; font-size: 1.05em; color: #202124;">UI: Business Ops × Customer Ops Chat</h2>
  <p style="margin: 7px 0; line-height: 1.6;">요청마다 한 개의 thread가 생성됩니다. Chat에는 team/agent 간 모든 메시지가 보이고, 오른쪽 trace에는 handoff envelope가 표시됩니다. fixture만 읽으며 외부 시스템을 변경하지 않습니다.</p>
</div>

In [ ]:
from pathlib import Path
import sys
import gradio as gr

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '7_Business_Operation_Harness').is_dir())
sys.path.insert(0, str(project_root / '7_Business_Operation_Harness'))
from src.cross_team_chat import interface_catalog, run_cross_team_chat

def event_as_chat_message(event):
    message = event['message']
    if message['from_team'] == 'user':
        return None
    header = f"**{message['from_team']} / {message['from_agent']} → {message['to_team']} / {message['to_agent']}**"
    return {'role': 'assistant', 'content': f"{header} — `{message['kind']}` · thread `{message['thread_id'][:8]}` — {message['content']}"}

def run_cross_team_ui(request, history):
    outcome = run_cross_team_chat(request)
    next_history = list(history or []) + [{'role': 'user', 'content': request}]
    next_history += [entry for event in outcome['trace'] if (entry := event_as_chat_message(event))]
    return '', next_history, outcome['summary'], outcome['verification'], outcome['trace']

with gr.Blocks() as cross_team_ui:
    gr.Markdown('## Cross-Team Deep Agent Interface')
    with gr.Row():
        with gr.Column():
            gr.Markdown('### Business Operations<br>Lead: `business_ops_lead` · Specialist: `inventory_specialist`')
        with gr.Column():
            gr.Markdown('### Customer Operations<br>Lead: `customer_ops_lead` · Specialist: `service_coordinator`')
    chatbot = gr.Chatbot(type='messages', label='Cross-team conversation', height=460)
    request = gr.Textbox(label='Operations request', value='SKU-A 공급 영향과 고객 안내 계획을 팀 간 조율해줘.')
    with gr.Row():
        send = gr.Button('Start cross-team chat', variant='primary')
        clear = gr.Button('Clear')
    final_summary = gr.Textbox(label='Shared final summary')
    with gr.Tab('Verification'):
        verification = gr.JSON(label='Thread and handoff checks')
    with gr.Tab('Message trace'):
        trace = gr.JSON(label='Delivered TeamMessage envelopes')
    with gr.Tab('Team interfaces'):
        gr.JSON(value=interface_catalog(), label='Accepted and emitted message kinds')
    send.click(run_cross_team_ui, inputs=[request, chatbot], outputs=[request, chatbot, final_summary, verification, trace])
    request.submit(run_cross_team_ui, inputs=[request, chatbot], outputs=[request, chatbot, final_summary, verification, trace])
    clear.click(lambda: ([], '', None, None), outputs=[chatbot, final_summary, verification, trace])
cross_team_ui.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', height=980)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124;">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 체크포인트</div>
  <ul style="margin: 8px 0 0; padding-left: 1.3em; line-height: 1.6;">
    <li>Business Ops와 Customer Ops의 내부 delegation이 각각 표시된다.</li>
    <li><code>team_handoff</code>와 <code>cross_team_commitment</code>가 양 팀 사이를 오간다.</li>
    <li>Verification에서 같은 thread ID, cross-team handoff 2건, external write false를 확인한다.</li>
  </ul>
</div>

---

## 선택 심화: team interface를 코드로 검증

각 Deep Agent team은 단순한 자연어 prompt가 아니라 수신·송신 가능한 `kind`와 payload 계약을 가져야 합니다. 다른 team은 lead agent의 공개 interface만 사용하고, 내부 specialist는 직접 호출하지 않습니다.

In [ ]:
interface_catalog()

### 1. 하나의 cross-team thread 실행

Business Ops lead는 재고 specialist에게만 일을 위임합니다. 근거 packet을 받은 뒤 Customer Ops lead에게 `team_handoff`를 보냅니다. Customer Ops도 자체 specialist를 거쳐 계획을 만들고 `cross_team_commitment`로 응답합니다.

In [ ]:
outcome = run_cross_team_chat('SKU-A 공급 영향과 고객 안내 계획을 팀 간 조율해줘.')
outcome['summary']

### 2. 실제 dispatch와 안전 경계 검증

이 assertion은 미리 만든 출력값을 비교하는 것이 아니라 message bus가 모든 handler를 실행한 결과의 path를 검사합니다. 실제 원격 DeepAgents/A2A transport를 연결할 때에도 동일한 thread와 권한 검증을 적용합니다.

In [ ]:
verification = outcome['verification']
assert verification['passed']
assert verification['same_thread_id']
assert verification['cross_team_handoffs'] == 2
assert verification['external_write'] is False
verification

In [ ]:
for event in outcome['trace']:
    message = event['message']
    print(f"{message['from_team']}/{message['from_agent']} → {message['to_team']}/{message['to_agent']} | {message['kind']}")